In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from black_scholes import bs_call, bs_put
from implied_vol import implied_vol

os.makedirs('../figures', exist_ok=True)
sns.set_theme(style='darkgrid')
print('Setup OK')


Setup OK


# Volatilité implicite et Volatility Smile

Ce notebook explore deux aspects :

1. **Validation** du solveur `implied_vol` : round-trip sur plusieurs niveaux de sigma.
2. **Smile de volatilité** : calcul de la vol implicite sur toute une chaîne d'options SPY
   et tracé de la courbe IV en fonction du moneyness.


## 1. Validation — round-trip sur plusieurs sigma

On price chaque option avec Black-Scholes à sigma connu, puis on réinjecte le prix
dans `implied_vol` et on vérifie qu'on retrouve la valeur de départ à 1e-6 près.


In [2]:
S, K, T, r = 100.0, 100.0, 1.0, 0.05
sigmas_test = [0.10, 0.20, 0.40]

rows = []
for sigma_true in sigmas_test:
    for opt in ('call', 'put'):
        price = bs_call(S, K, T, r, sigma_true) if opt == 'call' else bs_put(S, K, T, r, sigma_true)
        sigma_rec = implied_vol(price, S, K, T, r, opt)
        err = abs(sigma_rec - sigma_true)
        rows.append({'sigma_vrai': sigma_true, 'type': opt,
                     'prix_BS': price, 'sigma_implicite': sigma_rec,
                     'ecart': err, 'ok': err < 1e-6})

df_val = pd.DataFrame(rows)
print(df_val.to_string(index=False, float_format=lambda x: f'{x:.8f}'))
print(f"\nTous convergés < 1e-6 : {df_val['ok'].all()}")


 sigma_vrai type     prix_BS  sigma_implicite      ecart   ok
 0.10000000 call  6.80495771       0.10000000 0.00000000 True
 0.10000000  put  1.92790016       0.10000000 0.00000000 True
 0.20000000 call 10.45058357       0.20000000 0.00000000 True
 0.20000000  put  5.57352602       0.20000000 0.00000000 True
 0.40000000 call 18.02295145       0.40000000 0.00000000 True
 0.40000000  put 13.14589390       0.40000000 0.00000000 True

Tous convergés < 1e-6 : True


## 2. Smile de volatilité — données de marché SPY

On récupère la chaîne d'options SPY (premier expiry disponible > 20 jours) via `yfinance`.
Pour chaque strike, on calcule la vol implicite du call ou du put mid-price,
puis on trace IV en fonction du moneyness K/S.


In [3]:
import yfinance as yf
from datetime import datetime, timedelta

TICKER = 'SPY'
USE_SYNTHETIC = False   # sera mis à True si yfinance échoue

try:
    tk = yf.Ticker(TICKER)
    S0 = tk.fast_info['lastPrice']
    if S0 is None or S0 <= 0:
        raise ValueError('Prix spot indisponible')

    # Première maturité > 20 jours
    expirations = tk.options
    today = datetime.today()
    expiry = next(e for e in expirations
                  if (datetime.strptime(e, '%Y-%m-%d') - today).days > 20)
    T_exp = (datetime.strptime(expiry, '%Y-%m-%d') - today).days / 365.0

    chain = tk.option_chain(expiry)
    calls = chain.calls.copy()
    puts  = chain.puts.copy()

    # Mid-price
    calls['mid'] = (calls['bid'] + calls['ask']) / 2
    puts['mid']  = (puts['bid']  + puts['ask'])  / 2

    print(f'Ticker : {TICKER}  |  Spot : {S0:.2f}  |  Expiry : {expiry}  |  T = {T_exp:.4f} an(s)')
    print(f'Calls disponibles : {len(calls)}  |  Puts disponibles : {len(puts)}')

except Exception as exc:
    print(f'[yfinance indisponible] {exc}')
    USE_SYNTHETIC = True


Ticker : SPY  |  Spot : 741.00  |  Expiry : 2026-07-24  |  T = 0.0630 an(s)
Calls disponibles : 169  |  Puts disponibles : 162


In [4]:
if USE_SYNTHETIC:
    print('=== DONNÉES SYNTHÉTIQUES (yfinance inaccessible) ===')
    print('On génère une chaîne fictive avec un skew imposé pour démontrer le concept.\n')

    S0     = 500.0
    T_exp  = 45 / 365
    r_rate = 0.05
    expiry = 'synthétique'

    # Strikes autour de ATM
    strikes = np.arange(440, 561, 5, dtype=float)
    moneyness = strikes / S0

    # Smile synthétique : vol de base 0.18, skew négatif + convexité (smirk typique SPY)
    sigma_true = 0.18 + 0.08 * (1 - moneyness) + 0.04 * (1 - moneyness)**2

    # Génère les prix calls mid à partir de la vol synthétique
    call_prices = np.array([bs_call(S0, K, T_exp, r_rate, s)
                             for K, s in zip(strikes, sigma_true)])
    calls = pd.DataFrame({'strike': strikes, 'mid': call_prices})
    puts  = pd.DataFrame({'strike': strikes, 'mid': np.nan})  # non utilisé

    print(f'Strikes générés : {len(strikes)}  |  Vol range : [{sigma_true.min():.2%}, {sigma_true.max():.2%}]')
else:
    r_rate = 0.05   # taux sans risque approximatif


## 3. Calcul de la volatilité implicite par strike

Pour les strikes en dehors de la monnaie côté call (K > S), on peut utiliser le call ou le put.
Ici on utilise les **calls** sur toute la chaîne pour la simplicité, en filtrant les strikes
illiquides (bid–ask spread trop large, mid nul, ou IV hors de [0.01, 2.0]).


In [5]:
results = []
for _, row in calls.iterrows():
    K     = float(row['strike'])
    price = float(row['mid'])

    if pd.isna(price) or price <= 0:
        continue

    # Filtre liquidité : spread < 20 % du mid (uniquement si données réelles)
    if not USE_SYNTHETIC and 'bid' in row and 'ask' in row:
        spread = row['ask'] - row['bid']
        if spread > 0.20 * price:
            continue

    iv = implied_vol(price, S0, K, T_exp, r_rate, 'call')
    if np.isnan(iv) or iv < 0.01 or iv > 2.0:
        continue

    results.append({'strike': K, 'moneyness': K / S0, 'IV': iv, 'mid': price})

df_smile = pd.DataFrame(results).sort_values('moneyness')
print(f'Strikes après filtrage : {len(df_smile)}')
print(df_smile[['strike', 'moneyness', 'IV', 'mid']].to_string(
    index=False, float_format=lambda x: f'{x:.4f}'))


Strikes après filtrage : 149
  strike  moneyness     IV      mid
500.0000     0.6748 0.6596 242.8500
525.0000     0.7085 0.5988 217.9800
530.0000     0.7152 0.5840 212.9950
540.0000     0.7287 0.5692 203.0950
550.0000     0.7422 0.5371 193.1100
555.0000     0.7490 0.5254 188.1400
560.0000     0.7557 0.5136 183.1700
570.0000     0.7692 0.4899 173.2300
590.0000     0.7962 0.4429 153.3600
595.0000     0.8030 0.4307 148.3900
600.0000     0.8097 0.4196 143.4300
605.0000     0.8165 0.4071 138.4600
615.0000     0.8300 0.3847 128.5450
620.0000     0.8367 0.3735 123.5900
630.0000     0.8502 0.3510 113.6850
635.0000     0.8570 0.3392 108.7300
640.0000     0.8637 0.3287 103.7950
645.0000     0.8704 0.3036  98.6700
650.0000     0.8772 0.3075  93.9350
654.0000     0.8826 0.2992  90.0000
655.0000     0.8839 0.2970  89.0150
658.0000     0.8880 0.2909  86.0700
660.0000     0.8907 0.2869  84.1100
665.0000     0.8974 0.2765  79.2100
670.0000     0.9042 0.2668  74.3350
672.0000     0.9069 0.2629  72.3900

In [6]:
# Vol ATM = IV du strike le plus proche du spot
atm_row = df_smile.iloc[(df_smile['moneyness'] - 1.0).abs().argmin()]
iv_atm  = atm_row['IV']
print(f'Vol ATM (K={atm_row["strike"]:.1f}, K/S={atm_row["moneyness"]:.4f}) : {iv_atm:.4%}')


Vol ATM (K=741.0, K/S=1.0000) : 15.4439%


## 4. Le Volatility Smile / Skew

### Pourquoi la courbe n'est pas plate ?

Dans le modèle de Black-Scholes, la volatilité `sigma` est **constante** : tous les strikes
doivent avoir la même vol implicite. En pratique, la courbe IV vs strike est **inclinée ou
souriante** (smile), ce qui révèle que le marché attribue des prix différents selon le strike.

### Qu'est-ce qu'un skew ?

Sur les indices actions (SPY, CAC 40…), la courbe est typiquement un **skew négatif** :
- Les **puts OTM** (K < S) ont une vol implicite *plus élevée* que les calls OTM.
- Cela reflète une **prime de risque de krach** : les acheteurs de puts OTM paient cher
  une protection contre des baisses brutales (tail risk). Après le krach de 1987,
  ce phénomène est devenu structurel sur les marchés d'indices.

### En quoi cela contredit Black-Scholes ?

Black-Scholes suppose une distribution log-normale des rendements, symétrique et sans queues
épaisses. Or le marché *pricé* comme si la distribution réelle avait :
1. Une **asymétrie négative** (skewness < 0) : baisses plus probables que la lognormale.
2. Des **queues épaisses** (leptokurtosis) : événements extrêmes sous-estimés par BS.

La courbe IV non plate est la *signature* de ces déviations : c'est l'information que
le marché encode dans les prix d'options que BS ne peut pas capturer avec un seul sigma.


In [7]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(df_smile['moneyness'], df_smile['IV'] * 100,
        'o-', color='steelblue', lw=2, ms=5, label='Vol implicite (call)')
ax.axhline(iv_atm * 100, color='tomato', lw=1.5, ls='--',
           label=f'Vol ATM = {iv_atm:.2%}')
ax.axvline(1.0, color='grey', lw=1, ls=':', alpha=0.7)

label_src = 'synthétique' if USE_SYNTHETIC else 'marché'
ax.set_title(f'Volatility Skew — {TICKER}  |  Expiry {expiry}  |  données {label_src}',
             fontsize=13)
ax.set_xlabel('Moneyness  K / S')
ax.set_ylabel('Volatilité implicite (%)')
ax.legend()

# Annotation skew OTM
ax.annotate('Puts OTM : prime de risque\nde krach (skew négatif)',
            xy=(0.93, df_smile[df_smile['moneyness'] < 0.96]['IV'].max() * 100),
            xytext=(0.85, df_smile['IV'].max() * 100 * 0.85),
            arrowprops=dict(arrowstyle='->', color='tomato'),
            fontsize=9, color='tomato')

plt.tight_layout()
out_path = '../figures/07_volatility_smile.png'
plt.savefig(out_path, dpi=150)
plt.show()
print(f'Saved: figures/07_volatility_smile.png')


Saved: figures/07_volatility_smile.png


C:\Users\octav\AppData\Local\Temp\ipykernel_26388\3739631936.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Synthèse


In [8]:
summary = df_smile.copy()
summary['IV (%)'] = (summary['IV'] * 100).round(2)
summary['K/S']    = summary['moneyness'].round(4)
summary['OTM']    = summary['moneyness'].apply(
    lambda m: 'put OTM' if m < 0.98 else ('call OTM' if m > 1.02 else 'ATM'))

print(summary[['strike', 'K/S', 'IV (%)', 'OTM']].to_string(index=False))
print(f'\nVol ATM   : {iv_atm:.4%}')
print(f'Vol min   : {df_smile["IV"].min():.4%}  (K/S = {df_smile.loc[df_smile["IV"].idxmin(), "moneyness"]:.3f})')
print(f'Vol max   : {df_smile["IV"].max():.4%}  (K/S = {df_smile.loc[df_smile["IV"].idxmax(), "moneyness"]:.3f})')
print(f'Amplitude : {(df_smile["IV"].max() - df_smile["IV"].min())*100:.2f} vol points')


 strike    K/S  IV (%)      OTM
  500.0 0.6748   65.96  put OTM
  525.0 0.7085   59.88  put OTM
  530.0 0.7152   58.40  put OTM
  540.0 0.7287   56.92  put OTM
  550.0 0.7422   53.71  put OTM
  555.0 0.7490   52.54  put OTM
  560.0 0.7557   51.36  put OTM
  570.0 0.7692   48.99  put OTM
  590.0 0.7962   44.29  put OTM
  595.0 0.8030   43.07  put OTM
  600.0 0.8097   41.96  put OTM
  605.0 0.8165   40.71  put OTM
  615.0 0.8300   38.47  put OTM
  620.0 0.8367   37.35  put OTM
  630.0 0.8502   35.10  put OTM
  635.0 0.8570   33.92  put OTM
  640.0 0.8637   32.87  put OTM
  645.0 0.8704   30.36  put OTM
  650.0 0.8772   30.75  put OTM
  654.0 0.8826   29.92  put OTM
  655.0 0.8839   29.70  put OTM
  658.0 0.8880   29.09  put OTM
  660.0 0.8907   28.69  put OTM
  665.0 0.8974   27.65  put OTM
  670.0 0.9042   26.68  put OTM
  672.0 0.9069   26.29  put OTM
  674.0 0.9096   25.90  put OTM
  675.0 0.9109   25.71  put OTM
  677.0 0.9136   25.35  put OTM
  680.0 0.9177   24.82  put OTM
  681.0 